In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
import numpy as np

# 1. Load wildfire data (with latitude/longitude)
wildfire_df = pd.read_csv("../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv")

# 2. Construct proper startdate column
# Rename date columns to match what pd.to_datetime expects
wildfire_df = wildfire_df.rename(columns={
    'startdateyear': 'year',
    'startdatemonth': 'month',
    'startdateday': 'day'
})

# Construct proper datetime column
wildfire_df['startdate'] = pd.to_datetime(
    wildfire_df[['year', 'month', 'day']],
    errors='coerce'
)

# 3. Drop rows with missing coordinates
wildfire_df = wildfire_df.dropna(subset=['latitude', 'longitude'])

# 4. Set centroid lat/lon from those columns
wildfire_df['centroid_lat'] = wildfire_df['latitude']
wildfire_df['centroid_lon'] = wildfire_df['longitude']

# 5. Load airport locations
airport_df = pd.read_csv("../data/processed_data/airports_runways_joined.csv")
airport_df = airport_df.dropna(subset=['latitude_deg', 'longitude_deg'])

# 6. Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a)) * 1000  # meters

# 7. Match each wildfire to nearest airport
results = []

for _, fire in wildfire_df.iterrows():
    lat1, lon1 = fire['centroid_lat'], fire['centroid_lon']
    distances = haversine(lat1, lon1, airport_df['latitude_deg'], airport_df['longitude_deg'])
    min_idx = np.argmin(distances)
    nearest_airport = airport_df.iloc[min_idx]

    results.append({
        # Wildfire details
        'fire_id': fire['unique_id'],
        'fire_lat': lat1,
        'fire_lon': lon1,
        'startdate': fire['startdate'],
        'duration': fire['duration'],
        'size (acres)': fire['size (acres)'], 
        'fire_spread (acres/day)': fire['fire_spread (acres/day)'],  # same conversion

        # Distance
        'distance_nm': distances[min_idx] / 1852,  # in nautical miles

        # All airport details
        'ident': nearest_airport['ident'],
        'iata_code': nearest_airport['iata_code'],
        'icao_code': nearest_airport['icao_code'],
        'local_code': nearest_airport['local_code'],
        'closet_airport_name': nearest_airport['name'],
        'type': nearest_airport['type'],
        'latitude_deg': nearest_airport['latitude_deg'],
        'longitude_deg': nearest_airport['longitude_deg'],
        'elevation_ft': nearest_airport['elevation_ft'],
        'country_name': nearest_airport['country_name'],
        'region_name': nearest_airport['region_name'],
        'runway_lengths_ft': nearest_airport['runway_lengths_ft'],
        'runway_surfaces': nearest_airport['runway_surfaces'],
        'airtanker_base': nearest_airport['airtanker_base']
    })

# 6. Create DataFrame
result_df = pd.DataFrame(results)

# 7. Preview
print(result_df.head())
result_df.to_csv("../data/processed_data/nearest_airport_airtanker_bases_to_fires_final.csv", index=False)

      fire_id  fire_lat  fire_lon  startdate  duration  size (acres)  \
0  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
1  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
2  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
3  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
4  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   

   fire_spread (acres/day)  distance_nm ident iata_code  ...  \
0                 153.2547    18.608426  KCEW       CEW  ...   
1                 153.2547    18.608426  KCEW       CEW  ...   
2                 153.2547    18.608426  KCEW       CEW  ...   
3                 153.2547    18.608426  KCEW       CEW  ...   
4                 153.2547    18.608426  KCEW       CEW  ...   

  closet_airport_name            type latitude_deg longitude_deg  \
0   Bob Sikes Airport  medium_airport    30.778799    -86.522102   
1   Bob Sikes Airport  medium_airport    30.77

In [2]:
result_df.shape

(32082, 22)